In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 65. B11 Project — Treasury Curve Forecast-to-Decision Specification

## 学習目標


- economic hypothesis、information timestamp、target、baseline、cost、primary metricを1つの仕様へfreezeする。
- source・availability・methodology break・未識別のliquidity量を監査表へ残す。
- forecast、measure/control、exposure allocationを別artifactにし、winnerやproduction strategyを選ばない。


## 前提知識


- Week 41–44のCoreとB5–B10のreproducibility contract
- claim boundary、PIT、locked evaluation、quadratic allocation

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 65


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Final specification

Projectの成果物はmodel winnerではなく、次の7点を凍結したdecision specificationである。

1. economic hypothesisとinformation timestamp
2. official source、availability、methodology break
3. forecast targetとzero-change baseline
4. decision mappingとcurve exposure unit
5. observable liquidity fieldsとscenario-only costs
6. primary metric、candidate count、falsification
7. 不足するcapacity・crowding・funding・quote data

FINRA API access/terms gateが通らない限り、Week 42のreal-data claimはblockedであり、fixtureのidentity検算だけを提出する。

In [4]:
spec_rows = [
    {"item": "hypothesis/timestamp", "status": "frozen", "evidence": "Week 41 protocol"},
    {"item": "Treasury source/method break", "status": "frozen", "evidence": treasury.metadata.snapshot_sha256},
    {"item": "forecast target/baseline", "status": "frozen", "evidence": "5 publication observations / zero-change"},
    {"item": "FINRA aggregate", "status": "conditional", "evidence": "API access and terms gate not passed"},
    {"item": "quote-level execution cost", "status": "fixture-only", "evidence": "trade identity test"},
    {"item": "exposure allocation", "status": "frozen", "evidence": "yield-change units, no cash PnL"},
    {"item": "model selection", "status": "no_model_selected", "evidence": "specification project"},
]
spec_table = pd.DataFrame(spec_rows)
display(spec_table)
assert (spec_table.loc[spec_table["item"] == "model selection", "status"] == "no_model_selected").all()
assert (spec_table.loc[spec_table["item"] == "FINRA aggregate", "status"] == "conditional").all()

,item,status,evidence
0,hypothesis/timestamp,frozen,Week 41 protocol
1,Treasury source/method break,frozen,6ddef9605abbf02c6a4526a51f098135b41da1a4379156...
2,forecast target/baseline,frozen,5 publication observations / zero-change
3,FINRA aggregate,conditional,API access and terms gate not passed
4,quote-level execution cost,fixture-only,trade identity test
5,exposure allocation,frozen,"yield-change units, no cash PnL"
6,model selection,no_model_selected,specification project


In [5]:
evidence_order = ["source", "timestamp", "target", "baseline", "cost", "metric", "claim boundary"]
evidence_status = np.array([1, 1, 1, 1, 0, 1, 1], dtype=int)
evidence_frame = pd.DataFrame({"artifact": evidence_order, "frozen_or_available": evidence_status})
fig = go.Figure()
fig.add_bar(x=evidence_frame["artifact"], y=evidence_frame["frozen_or_available"], name="evidence")
fig.update_layout(title="B11 specification gate: unavailable cost data stays visible", xaxis_title="Artifact", yaxis_title="1=frozen / 0=conditional", template="plotly_white")
fig.show()
print("locked outer Treasury rows opened: False")
print("tradability claim allowed: False")

locked outer Treasury rows opened: False
tradability claim allowed: False


## 2. 失敗モード

- Projectで最良モデルを選ぶことを成果物にする。
- FINRA未承認のaggregateやfixtureを実約定データと呼ぶ。
- Treasury par yieldからzero curve、cash PnL、execution qualityを作る。
- 不足データを合成して、実証結果のように報告する。

## 3. 段階別演習

### 基礎

1. 7項目のspecificationを2ページのtechnical memoへ変換せよ。

### 標準

2. API access後に初めて開けるgateと、accessがなくても検証できるfixtureを分けよ。

### 研究

3. no-model-selectedを含むreplication packageのmanifest、hash、claim auditを設計せよ。

## 4. Exit Criteria

- [ ] 7つの仕様項目をfreezeした
- [ ] source、availability、methodology breakを記録した
- [ ] FINRAの条件付き状態を残した
- [ ] forecast/control/exposureの単位を分離した
- [ ] no-model-selectedとno-tradability-claimを明記した

## 5. 出典

- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)
- [U.S. Treasury Daily Treasury Par Yield Curve Rates](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve)

- [FINRA Treasury Daily Aggregate Statistics](https://www.finra.org/finra-data/browse-catalog/about-treasury)
- [FINRA Treasury Daily File](https://www.finra.org/finra-data/browse-catalog/about-treasury/daily-file)
- [FINRA Query API](https://developer.finra.org/products/query-api)
- [FINRA Fixed Income Data Specific Terms](https://developer.finra.org/specific-terms-fixed-income-data)

- [Fama and MacBeth (1973), Risk, Return, and Equilibrium](https://www.jstor.org/stable/1831028)
- [Hansen (1982), Large Sample Properties of GMM Estimators](https://doi.org/10.2307/1912775)
- [Duffie and Kan (1996), A Yield-Factor Model of Interest Rates](https://doi.org/10.1016/0304-405X(95)00881-6)
- [Boyd and Vandenberghe, Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/bv_cvxbook.pdf)